In [1]:
import warnings
warnings.filterwarnings('ignore')

### Import base libraries 

In [2]:
import re
import json
import os
import shutil
import time
from docx2pdf import convert
from pydantic import BaseModel
from crewai.flow import Flow, listen, start

#### Disable CrewAI telemetry (keeps local runs quieter and private)

In [3]:
os.environ['CREWAI_DISABLE_TELEMETRY'] = 'true'

### Logging

In [4]:
import logging

logging.basicConfig(
    filename='resume_extraction.log',  # Log file name
)

### Importing Agent Orchestestor Class

Below Agent Orchestrator calls two agents with corresponding tasks mentioned in the path: ph_resume_ext\src\ph_resume_ext\crews\resume_crew_pr\config

Two agents are:
1) Extractor_Agent
2) Processor_Agent

Segregation of Agents and Tasks help in clearly articulate LLM Prompts

Hyper parameters used with each agent in the orchestrator:
1) Temperature : to remain less creative while answering
2) top-k, top-p, max-tokens are not required

In [5]:
from ph_resume_ext.crews.resume_crew_pr.resume_crew import ResumeCrew

### Converting LLM's output (which are in JSON structure) are wrapped into JSON format by removing Markdown code blocks

In [6]:
def extract_json_from_markdown(text):
    """LLM's output in JSON structure are wrapped into JSON format by removing Markdown code blocks"""
    # Remove Markdown code block if present
    match = re.search(r"```(?:json)?\\s*([\\s\\S]*?)```", text, re.IGNORECASE)
    if match:
        return match.group(1)
    # Try again with real newlines (not escaped)
    match = re.search(r"```(?:json)?\s*([\s\S]*?)```", text, re.IGNORECASE)
    if match:
        return match.group(1)
    return text

### LLM response in JSON format are saved to file

In [7]:
def json_output(file_path,raw):
    """LLM response in JSON format are saved to file"""
    try:
        json_str = extract_json_from_markdown(raw)
        output = json.loads(json_str)
        logging.info(f"Successfully extracted JSON from LLM response for file: {file_path}")
    except (json.JSONDecodeError, TypeError):
        logging.error("Failed to decode JSON from LLM response.")
        print("Result was not JSON. Using raw text instead.")
        output = {"raw_text": raw}
    output_resume = r"resume\processed"
    base_name = os.path.splitext(os.path.basename(file_path))[0] 
    output_path = os.path.join(output_resume, base_name + ".json")
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=4)
    return output_path

### Find resume files and Store complete paths of each

In [8]:
def read_resume():
    """Find resume files and Store complete paths of each"""
    resume_paths = []
    resume = r"resume\templates"

    if os.path.exists(resume):
        logging.info(f"Reading resumes from directory: {resume}")
        for file_name in os.listdir(resume):
            file_path = file_name
            # file_path = os.path.join(resume, file_name)
            if os.path.isfile(file_path):
                resume_paths.append(file_name)
            else:
                resume_paths.append(file_path)
    return resume_paths

In [9]:
for resume_path in read_resume():
    print(resume_path)

elegant-ms-word-resume-template.docx
entry-level-data-scientist-resume-example.pdf
official-ms-word-resume-template.docx
senior-data-scientist-resume-example.pdf
standout-ms-word-resume-template.docx


### Process each resume file found and convert word to pdf files

In [10]:
def process_resumes():
    """Process each resume file found"""
    resume_processing_path_list = []
    for resume_file in  read_resume():
        resume = r"resume\templates"
        resume_path = os.path.join(resume, resume_file)
        resume_processing_path = os.path.join(r"resume\processing", resume_file)
        """Doc to PDF conversion"""
        format_type = os.path.splitext(resume_path)[1] 
        if format_type in ['.doc', '.docx']:
            logging.info(f"Converting {resume_path} to PDF format.")
            pdf_path = f"{os.path.splitext(resume_processing_path)[0]}.pdf"
            convert(resume_path, pdf_path)
            resume_processing_path = pdf_path
        elif format_type == '.pdf':
            logging.info(f"Copying PDF file {resume_path} to processing directory.")    
            shutil.copy2(resume_path, resume_processing_path)

        print("Found resume:", resume_processing_path)
        logging.info(f"Found resume: {resume_processing_path}")
        resume_processing_path_list.append(resume_processing_path)
    return resume_processing_path_list

In [11]:
print (process_resumes())

100%|██████████| 1/1 [00:03<00:00,  3.45s/it]


Found resume: resume\processing\elegant-ms-word-resume-template.pdf
Found resume: resume\processing\entry-level-data-scientist-resume-example.pdf


100%|██████████| 1/1 [00:03<00:00,  3.17s/it]


Found resume: resume\processing\official-ms-word-resume-template.pdf
Found resume: resume\processing\senior-data-scientist-resume-example.pdf


100%|██████████| 1/1 [00:03<00:00,  3.47s/it]

Found resume: resume\processing\standout-ms-word-resume-template.pdf
['resume\\processing\\elegant-ms-word-resume-template.pdf', 'resume\\processing\\entry-level-data-scientist-resume-example.pdf', 'resume\\processing\\official-ms-word-resume-template.pdf', 'resume\\processing\\senior-data-scientist-resume-example.pdf', 'resume\\processing\\standout-ms-word-resume-template.pdf']


### Crew of Agents and corresponding Tasks are executed on each resume

--> Reasoning reduces Hallucination. Therefore, with each LLM prompt, reasoning is provided. Complete prompt (including Reasoning) is demonstrated in the final processing

In [12]:
def execute_LLM():
    """Call ResumeCrew to extract information""" 
    resume_processing_path_list = process_resumes() 
    for file_path in resume_processing_path_list:
        time.sleep(5) # To avoid rate limiting
        logging.info(f"Processing resume: {file_path}")
        print("Processing resume:", file_path)  
        result = (
                        ResumeCrew()
                        .crew()
                        .kickoff(inputs={"file_path": file_path})
                    )

        # Extract JSON from Markdown if needed
        json_file = json_output(file_path, result.raw)
        logging.info(f"JSON output saved to: {json_file}")
        print("JSON output saved to:", json_file)
        time.sleep(5)
        print("-" * 50)

### Executing Full Code base & storing result at path: "ph_resume_ext\src\ph_resume_ext\resume\processed"

In [13]:
execute_LLM()

100%|██████████| 1/1 [00:03<00:00,  3.52s/it]


Found resume: resume\processing\elegant-ms-word-resume-template.pdf
Found resume: resume\processing\entry-level-data-scientist-resume-example.pdf


100%|██████████| 1/1 [00:01<00:00,  1.03s/it]


Found resume: resume\processing\official-ms-word-resume-template.pdf
Found resume: resume\processing\senior-data-scientist-resume-example.pdf


100%|██████████| 1/1 [00:03<00:00,  3.49s/it]


...

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Document Text Extractor                                                                                 │
│                                                                                                                 │
│  Thought: Okay, I will start by extracting the text from the PDF file using the PDF Reader tool.                │
│                                                                                                                 │
│  Using Tool: PDF Reader                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"file_path\": \"resume\\\\processing\\\\standout-ms-word-resume-template.pdf\", \"page_range\": \"all\"}"   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  PDF: standout-ms-word-resume-template.pdf                                                                      │
│  Total Pages: 1                                                                                                 │
│  Pages Read: all (1-1)                                                                                          │
│                                                                                                                 │
│  Document Information:                                                                                          │
│  - /Author: M Awais Lodhi                                                                                       │
│  - /Creator: Microsoft® Word 2024                                                                               │
│  - /CreationDate: D:20250905073808-04'00'                                                                       │
│  - /ModDate: D:20250905073808-04'00'                                                                            │
│  - /Producer: Microsoft® Word 2024                                                                              │
│                                                                                                                 │
│  --- Content Start ---                                                                                          │
│                                                                                                                 │
│  --- Page 1 ---                                                                                                 │
│  Text preview: Labor and Delivery Nurse  KAIDA KIM   Work Experience     EDUCATION   Bachelor of Science        │
│  Nursin...                                                                                                      │
│  Text length: 2296 characters                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  Labor and Delivery Nurse  KAIDA KIM                                                                            │
│  Work Experience                                                                                                │
│   EDUCATION                                                                                                     │
│  Bachelor of Science                                                                                            │
│  Nursing                                                                                                        │
│  Vanderbilt University                                                                                          │
│  2018 -2020                                                                                                     │
│  Nashville, TN                                                                                                  │
│  SKILLS                                                                                                         │
│  •McKesson                                                                                                      │
│  •GE Corometrics                                                                                                │
│  •Baxter Sigma Spectrum                                                                                         │
│  •Masimo Root with Radius -7                                                                                    │
│  •GE Healthcare CARESCAPE                             

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Document Text Extractor                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "status": "success",                                                                                         │
│    "First_Name": "Kaida",                                                                                       │
│    "Last_Name": "Kim",                                                                                          │
│    "email_address": "kim.kaida@gmail.com",                                                                      │
│    "skills": [                                                                                                  │
│      "McKesson",                                                                                                │
│      "GE Corometrics",                                                                                          │
│      "Baxter Sigma Spectrum",                                                                                   │
│      "Masimo Root with Radius -7",                                                                              │
│      "GE Healthcare CARESCAPE Monitor B650/B850",                                                               │
│      "Abbott i -STAT System",                                                                                   │
│      "Siemens epoc Blood Analysis System"                                                                       │
│    ],                                                                                                           │
│    "phone_number": "(123) 456-7890",                                                                            │
│    "location": "Knoxville, TN",                                                                                 │
│    "linkedin": null,                                                                                            │
│    "summary": null,                                                                                             │
│    "work_experience": {                                                                                         │
│      "East Tennessee Children's Hospital": {                                                                    │
│        "title": "Labor and Delivery Nurse",                                                                     │
│        "location": "Knoxville, TN",                                                                             │
│        "dates": "November 2019 - current",                                                                      │
│        "description": [                                                                                         │
│          "Used GE Corometrics fetal monitors to track and evaluate fetal heart rates for 450+ cases each year,  │
│  ensuring accurate and timely intervention as needed",                                                          │
│          "Developed and implemented individualized care plans for high -risk pregnancies, decreasing            │
│  complications by 77%",                                                                                         │
│          "Used the Siemens epoc Blood Analysis System to provide immediate blood gas, electrolyte, and          │
│  metabolite results",                                  

JSON output saved to: resume\processed\standout-ms-word-resume-template.json
--------------------------------------------------
